# Resonant cyclotron scattering

We study resonant cyclotron scattering and how it modifies a black-body spectrum at source. We assume the 1D model from [Lyutikov and Gavriil (2006)](https://ui.adsabs.harvard.edu/abs/2006MNRAS.368..690L/abstract) (see also [Rea et al. 2008](https://ui.adsabs.harvard.edu/abs/2008ApJ...686.1245R/abstract)).

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scipy.integrate as integrate
import scipy.special as scsp

from scipy.integrate import trapz, quad
import pypopsyn.simulator.basics.constants as const
import pypopsyn.simulator.multiband_emission.emission_xray as xem
import pypopsyn.simulator.interstellar_medium.nh_model as nhm
import pypopsyn.simulator.interstellar_medium.xray_abs_cross_section as xabs
import utilities.plot_settings

In [ ]:
def dirac_delta(x: np.ndarray) -> np.ndarray:
    """
    Approximation of the Dirac delta function. This approximation is used for plotting purposes.

    Args:
        x (np.ndarray): Array of values.

    Returns:
        (np.ndarray): Value of the Dirac delta function.
    """
    epsilon = 1e-10

    # The value of the Dirac delta inside the interval [-epsilon, epsilon] is chosen to be 2.0e2 to have a 
    # appropriate visulaization of the n+ function.
    dirac_delta = np.where(np.abs(x) < epsilon, 2.e2, 0.0)

    return dirac_delta


def n_plus_eta(
    eta: np.ndarray,
    tau_0: np.ndarray,
    beta_T: np.ndarray,
) -> np.ndarray:
    """
    Transmission function n+ in eq. (35) in Lyutikov and Gavriil (2006).
    This uses an approximate Dirac delta function for visualization purposes.

    Args:
        eta (np.ndarray): Array of (E-E_0)/E_0 values for the reflected photons where E_0 and E are the initial and 
            final photon energies respectively.
        tau_0 (np.ndarray): Array of optical depths tau_0 (see eq. 2 in Lyutikov and Gavriil 2006).
        beta_T (np.ndarray): Array of thermal velocities for the electrons/positrons in units of the speed of light.

    Returns:
        (np.ndarray): Value of the transmission probability n+.
            This will have shape (NS_number, len(E), len(E_0)).
    """

    # Reshape the input arrays to make it compatible for broadcasting.
    eta = eta[np.newaxis, :]
    tau_0 = tau_0[:, np.newaxis]
    beta_T = beta_T[:, np.newaxis]

    term_1 = (
        tau_0
        / (8.0 * beta_T)
        * ((4.0 * beta_T - eta) / eta) ** 0.5
    )
    I1 = scsp.i1(
        tau_0
        / (4.0 * beta_T)
        * (xi * (4.0 * beta_T - eta)) ** 0.5
    )

    term_2 = term_1 * I1
    term_2 = np.nan_to_num(term_2, nan=0)

    n_p = np.exp(-tau_0 / 2.0) * (dirac_delta(eta) + term_2)

    return n_p

def n_minus_xi(
    xi: np.ndarray,
    tau_0: np.ndarray,
    beta_T: np.ndarray,
) -> np.ndarray:
    """
    Reflection function n- in eq. (35) in Lyutikov and Gavriil (2006).
    Note that in the original paper this equation misses a factor 1/2 and in the exponential term should be tau_0/2 instead of tau_0.

    Args:
        xi (np.ndarray): Array of (E_0-E)/E_0 values for the reflected photons where E_0 and E are the initial and final photon 
            energies respectively.
        tau_0 (np.ndarray): Array of optical depths tau_0 (see eq. (2) in Lyutikov and Gavriil 2006).
        beta_T (np.ndarray): Array of thermal velocities for the electrons/positrons in units of the speed of light.

    Returns:
        (np.ndarray): Value of the transmission probability n-.
            This will have shape (NS_number, len(E), len(E_0)).
    """

    # Reshape omega to make it compatible for broadcasting.
    xi = xi[np.newaxis, :]
    tau_0 = tau_0[:, np.newaxis]
    beta_T = beta_T[:, np.newaxis]

    I0 = scsp.i0(
        tau_0
        / (4.0 * beta_T)
        * ((2.0 * beta_T - xi) * (xi + 2.0 * beta_T)) ** 0.5
    )

    n_m = tau_0 / (8.0 * beta_T) * np.exp(-tau_0 / 2.0) * I0
    n_m = np.nan_to_num(n_m, nan=0)

    return n_m

In [ ]:
def resonant_cyclotron_scat_spectrum_reflections(
    E: np.ndarray,
    E_0: np.ndarray,
    tau_0: np.ndarray,
    beta_T: np.ndarray,
    I_ph_source: np.ndarray,
    n_reflections: int,
) -> np.ndarray:
    """
    Compute the spectrum resulting from different reflections and transmissions due to resonant cyclotron scattering (RCS)
    given a source intensity spectrum (see Lyutikov and Gavriil 2006). 
    This function allows to show the subsequent approximations of the final spectrum given by each reflection.

    Args:
        E (np.ndarray): Array of energies in [erg] of the transmitted intensity.
        E_0 (np.ndarray): Array of energies in [erg] of the source intensity.
        tau_0 (np.ndarray): Array of optical depths tau_0 (see eq. 2 in Lyutikov and Gavriil 2006).
        beta_T (np.ndarray): Array of thermal velocities for the electrons/positrons in units of the speed of light.
        I_ph_source (np.ndarray): Intensity of the source in [ph cm^-2 s^-1 erg^-1 sterad^-1].
        n_reflections (int): Number of reflections (6 reflections guarantees convergence of the final spectrum,
            see Lyutikov and Gavriil 2006).

    Returns:
        (np.ndarray): Resonant cyclotron scattering spectrum intensity in [ph cm^-2 s^-1 erg^-1 sterad^-1].
    """
    rcs_spectrum = np.zeros((len(tau_0), n_reflections + 1, len(E)))

    # Compute the transmission and reflection probabilities.
    n_trans_without_delta = xem.n_plus_without_delta(E, E_0, tau_0, beta_T)
    exp_fact = np.exp(- tau_0 / 2.0)
    exp_fact = exp_fact[:, np.newaxis]
    p_refl = xem.n_minus(E, E_0, tau_0, beta_T)

    # Compute the RCS spectrum by considering multiple reflections and transmissions.
    # (see eq. 42 in Lyutikov and Gavriil 2006).
    I_ph = I_ph_source[:, np.newaxis, :]
    I_ph_trans = I_ph_source * exp_fact + trapz((I_ph * n_trans_without_delta), E_0, axis=2)
    rcs_spectrum[:, 0, :] = rcs_spectrum[:, 0, :] + I_ph_trans

    for i in range(1, n_reflections + 1):
        I_ph_reflect = trapz((I_ph * p_refl), E_0, axis=2)
        I_ph_reflect_reshape = I_ph_reflect[:, np.newaxis, :]
        I_ph_trans_refl = I_ph_reflect * exp_fact + trapz((I_ph_reflect_reshape * n_trans_without_delta), E_0, axis=2)
        rcs_spectrum[:, i, :] = rcs_spectrum[:, i-1, :] + I_ph_trans_refl

        I_ph = I_ph_trans_refl[:, np.newaxis, :]

    return rcs_spectrum

For this example we consider a black-body spectrum at source with $k_{\rm B} T = 1$ keV, an average plasma thermal velocity $\beta_{\rm T} = 0.3$ and resonant optical depth values of $\tau_{\rm res} = 0.2, 1, 2, 6, 10, 20$ to reproduce the first plot in Fig. 2 in [Gullon et al. (2015)](https://ui.adsabs.harvard.edu/abs/2015MNRAS.454..615G/abstract).

In [ ]:
T = np.array([1000 * const.EV_TO_ERG / const.K_B])
tau_res = np.array([0.2, 1., 2., 6., 10., 20.])
beta_T = np.array([0.3, 0.3, 0.3, 0.3, 0.3, 0.3])
tau_0 = tau_res / 2.

# Define the array of energies between 10 eV and 20 KeV.
E_0 = np.logspace(0.5, np.log10(20000), 1000)
E = E_0

# Convert energies to eV.
E_0_erg = E_0 * const.EV_TO_ERG
E_erg = E * const.EV_TO_ERG

# Define the array of xi and eta values for the transmitted and reflected photons.
eta = np.linspace(-2, 2, 1001)
xi = np.linspace(-2, 2, 1001)

# Index to select a specific value of tau_res, choose between 0 and 5.
index = 5

First let's plot the transmission and reflection probabilities, $n_{+}$ and $n_{-}$ as a function of $(E - E_0) / E_0$ to reproduce a plot similar to Fig. 2 in [Lyutikov and Gavriil (2006)](https://ui.adsabs.harvard.edu/abs/2006MNRAS.368..690L/abstract).

In [ ]:
# We use here a n_plus function with an approximated Dirac delta for visualization purposes.
p_trans_eta = n_plus_eta(eta, tau_0, beta_T)
p_refl_xi = n_minus_xi(xi, tau_0, beta_T)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

#ax.set_xscale('log') 
#ax.set_yscale('log')
ax.set_xlim(-1.0,1.5) 
ax.set_ylim(0.,1.) 
ax.set_xlabel(r'$(E - E_0) / E_0$')
ax.set_ylabel(r'$\mathcal{P}$')

ax.plot( 
    eta,
    p_trans_eta[index, :],
    linestyle='-',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"Transmission probability",
)
ax.plot( 
    xi,
    p_refl_xi[index, :],
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"Reflection probability",
)

plt.legend(frameon=False, loc=0)
plt.grid()

In [ ]:
p_refl = xem.n_minus(E_erg, E_0_erg, tau_0, beta_T)
p_trans_without_delta = xem.n_plus_without_delta(E_erg, E_0_erg, tau_0, beta_T)

# Compute the total transmission and reflection probabilities as a function of the initial photon energy E_0 and 
# compare it with the expected probability.
# Transmission probability (we perform the analytical integral for the Dirac delta term and the numerical integral for the second term):
exp_fact = np.exp(- tau_0 / 2.0)
exp_fact = exp_fact[:, np.newaxis]
p_trans_tot = exp_fact + trapz(p_trans_without_delta, E_erg, axis=1)

# Reflection probability:
p_refl_tot = trapz(p_refl, E_erg, axis=1)

# Total expected transmission and reflection probabilities (eq. 37 in Lyutikov and Gavriil 2006).
p_refl_tot_expected = (1. - np.exp(-tau_0))/2.
p_trans_tot_expected = 1. - p_refl_tot_expected

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

ax.set_xscale('log') 
#ax.set_yscale('log')
#ax.set_xlim(0.2,2.2) 
#ax.set_ylim(0.,2.e-18) 
ax.set_xlabel(r'$E_0$ [eV]')
ax.set_ylabel(r'$n_{+} \, (n_{-})$')

ax.plot( 
    E_0,
    p_trans_tot[index, :].flatten(),
    linestyle='-',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"$\mathcal{P}_{\rm trans}$",
)
ax.plot( 
    E_0,
    p_trans_tot_expected[index] * np.ones(len(E_0)),
    linestyle='-',
    linewidth=4,
    color="tab:red",
    rasterized=True,
    label=r"expected $\mathcal{P}_{\rm trans}$",
)
ax.plot( 
    E_0,
    p_refl_tot[index, :].flatten(),
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"$\mathcal{P}_{\rm refl}$",
)
ax.plot( 
    E_0,
    p_refl_tot_expected[index] * np.ones(len(E_0)),
    linestyle='--',
    linewidth=4,
    color="tab:red",
    rasterized=True,
    label=r"expected $\mathcal{P}_{\rm refl}$",
)

plt.legend(frameon=False, loc=0)
plt.grid()

Compute the RCS spectrum.

In [ ]:
I_source = xem.blackbody_intensity_spectrum(E_0_erg, T)
I_ph_source = I_source / E_0_erg
I_rcs = xem.resonant_cyclotron_scat_spectrum(
    E, 
    E_0, 
    tau_0, 
    beta_T, 
    I_ph_source, 
    n_reflections=6
) * E_0_erg

I_rcs_reflections = resonant_cyclotron_scat_spectrum_reflections(
    E, 
    E_0, 
    tau_0, 
    beta_T, 
    I_ph_source, 
    n_reflections=6
) * E_0_erg

# Convert the spectra from [erg cm^-2 s^-1 erg^-1 sterad^-1] to [erg cm^-2 s^-1 eV^-1 sterad^-1].
I_source = I_source * const.EV_TO_ERG
I_rcs = I_rcs * const.EV_TO_ERG
I_rcs_reflections = I_rcs_reflections * const.EV_TO_ERG

This plot shows that after 6 reflection + transmission contributions the spectrum already converges quite well.

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

#ax.set_xscale('log') 
#ax.set_yscale('log')
ax.set_xlim(0.1,10.) 
#ax.set_ylim(1.e-1,1.e20) 
ax.set_xlabel(r'Energy [keV]')
ax.set_ylabel(r'$I(E)$ [erg cm$^{-2}$ s$^{-1}$ eV$^{-1}$ sterad$^{-1}$]')

ax.plot( 
    E*1.e-3,
    I_source.flatten(),
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"BB intensity",
)
# Plot the RCS spectrum for progressively increasing number of reflections denoted by the index i.
for i in range(I_rcs_reflections.shape[1]):
    ax.plot( 
        E*1.e-3,
        I_rcs_reflections[index, i, :],
        linestyle='-',
        linewidth=4,
        color="tab:red",
        alpha=0.3,
        rasterized=True,
    )
ax.plot( 
    E*1.e-3,
    I_rcs[index, :].flatten(),
    linestyle='-',
    linewidth=4,
    color="tab:red",
    rasterized=True,
    label=r"RCS spectrum",
)
plt.legend(frameon=False, loc=0)
plt.grid()

The following plot tries to reproduce the first plot in Fig. 2 in [Gullon et al. (2015)](https://ui.adsabs.harvard.edu/abs/2015MNRAS.454..615G/abstract). The units are different since we are plotting the intensity at the source not the flux observed on Earth. Also notice that in their legend they are mistakenly reporting the values of $\tau_0$ instead of $\tau_{\rm res}$.

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(tau_res))
norm = mpl.colors.Normalize(vmin=np.min(tau_res), vmax=np.max(tau_res)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

#ax.set_xscale('log') 
#ax.set_yscale('log')
ax.set_xlim(0.1,10.) 
#ax.set_ylim(1.e-1,1.e20) 
ax.set_xlabel(r'Energy [keV]')
ax.set_ylabel(r'$I(E)$ [erg cm$^{-2}$ s$^{-1}$ eV$^{-1}$ sterad$^{-1}$]')

ax.plot( 
    E*1.e-3,
    I_source.flatten(),
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"BB intensity",
)
for i in range(len(tau_res)):
    ax.plot( 
        E*1.e-3,
        I_rcs[i, :].flatten(),
        linestyle='-',
        linewidth=4,
        color=colors(i),
        rasterized=True,
        label=fr"$\tau_{{\mathrm{{res}}}} = {tau_res[i]}$",
    )

plt.legend(frameon=False, loc=0)
plt.grid()